# p162 Find Peak Element 解析笔记

- 题号：p162
- 题目英文名：Find Peak Element
- 题目中文名：寻找峰值
- 题目链接：https://leetcode.cn/problems/find-peak-element/
- 题型：二分查找
- 难度：Medium
- 推荐优先级：中
- Java 原代码位置：`solutions-java/leetcode/p162_find_peek_element/FindPeekElement.java`


## 1. 题目一句话总结

这道题要求我们在数组中用 `O(log n)` 时间找到任意一个峰值位置。

本质上考的是如何基于局部单调关系做二分，而不是传统意义上只在全局有序数组里二分。


## 2. 题目理解与约束分析

### 2.1 题目要求

给定一个整数数组 `nums`，其中相邻元素保证不相等。我们要找到一个峰值元素的位置。

峰值元素的定义是：严格大于左右相邻元素。

### 2.2 输入与输出

- 输入：整数数组 `nums`
- 输出：任意一个峰值下标
- 返回结果含义：只要返回的是某个合法峰值位置即可

### 2.3 关键约束

- 相邻元素不相等
- 题目要求时间复杂度为 `O(log n)`
- 可以把数组边界外视作负无穷

### 2.4 示例分析

输入：`[1,2,3,1]`

其中 `3` 同时大于左右两边的 `2` 和 `1`，所以峰值下标是 `2`。


## 3. Java 原代码

```java
package leetcode.p162_find_peek_element;

public class FindPeekElement {

    public static int findPeak(int[] nums) {
        if (nums == null || nums.length == 0) {
            return -1;
        }
        if (nums.length == 1) {
            return 0;
        }
        if (nums[0] > nums[1]) {
            return 0;
        }
        if (nums[nums.length - 1] > nums[nums.length - 2]) {
            return nums.length - 1;
        }
        int left = 1;
        int right = nums.length - 2;
        while (left <= right) {
            int middle = left + (right - left) / 2;
            if (nums[middle] > nums[middle - 1] && nums[middle] > nums[middle + 1]) {
                return middle;
            } else if (nums[middle] > nums[middle - 1]) {
                left = middle + 1;
            } else {
                right = middle - 1;
            }
        }
        return -1;
    }
}
```


## 4. 先从 Java 原方案理解

Java 原方案不是一上来就整段二分，而是先把几种边界情况单独处理掉：

- 空数组：返回 `-1`
- 长度为 1：唯一元素就是峰值
- 第一个元素比第二个大：峰值在下标 `0`
- 最后一个元素比倒数第二个大：峰值在最后位置

这样处理完之后，二分区间就可以安全缩到 `[1, n-2]`，因为我们知道真正要查找的峰值一定出现在中间范围内。

接着 Java 用中点附近的局部关系判断往哪边找：

- 如果 `middle` 已经同时大于左右两边，那它就是峰值
- 如果 `middle > middle-1`，说明这里处于上升趋势，峰值一定在右边
- 否则统一往左边找


## 5. 从朴素思路到优化思路

### 5.1 最直接的想法

最直观的做法是线性扫描，逐个判断某个位置是否大于左右邻居。

### 5.2 为什么不够好

- 线性扫描是 `O(n)`
- 题目明确要求 `O(log n)`

### 5.3 优化方向

因为相邻元素不相等，所以数组局部一定会呈现上升或下降关系。Java 原方案正是利用这种局部趋势来排除一半区间，从而做二分。


## 6. 核心算法讲解

### 6.1 算法名称

- 二分查找
- 局部趋势判断

### 6.2 为什么想到这个算法

因为题目不要求找最大值，只要求找任意一个峰值。只要能通过局部关系判断“哪一侧一定存在峰值”，就可以像二分一样不断缩小范围。

### 6.3 关键状态

- `left`、`right`：二分区间边界
- `middle`：当前中点
- `nums[middle-1]`、`nums[middle]`、`nums[middle+1]`：用于判断局部形态

### 6.4 步骤拆解

1. 先处理边界和极小规模情况
2. 把搜索区间放到中间部分 `[1, n-2]`
3. 取中点 `middle`
4. 如果 `middle` 比左右都大，直接返回
5. 如果 `middle` 大于左边，说明右侧有机会出现峰值，向右二分
6. 否则向左二分


## 7. 过程演示

以 `nums = [1,2,3,1]` 为例：

```text
边界检查：
nums[0] > nums[1] ? 否
nums[3] > nums[2] ? 否

进入二分区间：left = 1, right = 2

middle = 1
nums[1] = 2
2 > 1 成立，但 2 > 3 不成立
说明还在上升区间，往右找

left = 2, right = 2
middle = 2
nums[2] = 3，且 3 > 2 并且 3 > 1
找到峰值，下标为 2
```


In [ ]:
class Solution:
    def findPeakElement(self, nums: list[int]) -> int:
        if not nums:
            return -1
        if len(nums) == 1:
            return 0
        if nums[0] > nums[1]:
            return 0
        if nums[-1] > nums[-2]:
            return len(nums) - 1

        left, right = 1, len(nums) - 2
        while left <= right:
            middle = left + (right - left) // 2
            if nums[middle] > nums[middle - 1] and nums[middle] > nums[middle + 1]:
                return middle
            elif nums[middle] > nums[middle - 1]:
                left = middle + 1
            else:
                right = middle - 1

        return -1


## 8. 代码逐段讲解

### 8.1 边界处理

Java 原解专门先处理了空数组、长度为 1、左右边界直接成峰值的情况。这样进入主循环后，`middle-1` 和 `middle+1` 的访问就都安全了。

### 8.2 主二分逻辑

如果中点不是峰值，那么 Java 原解会用 `nums[middle]` 和 `nums[middle-1]` 的关系判断方向。

- 若 `middle` 比左边大，说明当前位置还在上升，右边一定存在峰值
- 否则统一往左找

### 8.3 为什么一定能找到

因为边界外可以视作负无穷，而相邻元素又不相等，所以数组不可能一直单调但没有峰值。每次二分排除掉的一半区间都不会错失所有峰值。


## 9. 复杂度分析

- 时间复杂度：`O(log n)`
- 为什么是这个时间复杂度：每次都排除一半区间
- 空间复杂度：`O(1)`
- 为什么是这个空间复杂度：只使用了常数个变量


## 10. 易错点

1. 没有先处理边界，就直接访问 `middle-1` 或 `middle+1`。
2. 误以为一定要找全局最大值，其实任意峰值都可以。
3. 没利用“相邻元素不相等”这个重要条件，导致二分方向判断不稳。
4. 把这题写成线性扫描，虽然能做对，但不满足复杂度要求。


## 11. 面试时怎么讲

可以这样概括：

> 这题我会用二分，不是因为数组整体有序，而是因为局部上升下降关系足以决定搜索方向。
> 我会先处理边界峰值，然后在中间区间二分。
> 如果中点已经比左右都大，直接返回；如果中点比左边大，说明右侧存在峰值，否则左侧存在峰值。
> 所以每次都能安全排掉一半区间，时间复杂度是 `O(log n)`。


## 12. 实际应用场景

这题可以类比成：在局部波动数据中快速找到一个局部极大点，而不是全局扫描。

### 具体业务案例：监控曲线局部峰值定位

#### 业务背景

监控系统里常常会保存一段资源使用率曲线，比如 CPU、QPS、延迟等。

#### 输入是什么

输入是一段时间序列数组。

#### 算法介入点

系统想快速找到一个明显峰值位置做告警或进一步分析，而不是每次都扫完整段数据。

#### 输出是什么

输出是某个局部峰值的下标。

#### 价值是什么

它解决的是局部极值快速定位问题。


In [ ]:
print('示例一:', Solution().findPeakElement([1, 2, 3, 1]))
print('示例二:', Solution().findPeakElement([1, 2, 1, 3, 5, 6, 4]))
print('单元素:', Solution().findPeakElement([10]))


## 13. Demo 输出说明

- 第一组通常输出 `2`，因为值 `3` 是明显峰值。
- 第二组可能输出 `1` 或 `5`，因为题目允许返回任意一个峰值。
- 第三组输出 `0`，因为唯一元素自然是峰值。


## 14. 一句话复盘

> 这题最关键的不是数组有序，而是像 Java 原解那样用局部趋势替代全局有序，照样完成二分。
